## Step 1: Bronze Layer — CSV → Delta Lake

**Bronze 层的职责**: 从数据源读取原始 CSV，无损贴源写入 Delta Lake。

**设计原则**:
- 只做格式转换（CSV → Delta），不做任何清洗、去重、Join
- 使用 `inferSchema` 自动推断列类型，不手动 CAST（故意保留原始数据面貌）
- 以 Delta 格式存储，获得 ACID 事务 + 时间旅行 + Schema 演化能力
- 清洗和标准化留给 Silver 层

**数据源说明**（面试必读）:
- 🏭 **生产架构**: 原始 CSV 上传到 GCS bucket (`gs://yuto-olist_raw_data/`),
  集群通过绑定的 GCP Service Account 自动认证读取。
- 🧪 **当前 Demo 实现**: 受限于 Databricks Serverless free tier 的 JVM 沙箱限制
  (`_jsc.hadoopConfiguration()` 被禁用，无法配置 GCS 认证)，
  本 Notebook 使用 Databricks Volumes 作为数据入口。
- 📌 **两者区别仅在于数据源路径** — 读取逻辑、Delta 写入方式完全一致。
  若切换到 Dedicated 集群或 Production 环境，只需将下方 `VOLUME_BASE` 替换为 `GCS_BUCKET`，
  并配置 `hadoopConfiguration` 即可无缝切换。

**输出**: 9 张 Delta 表，存入 Databricks hive_metastore 的 `default` schema

In [0]:
# ============================================================
# 配置区：定义数据源路径和 9 张表的映射关系
# ============================================================

# 数据源根路径 — Databricks Volumes
# 生产环境切换方式：将这行替换为 GCS_BUCKET = "gs://yuto-olist_raw_data"
# 并配置 spark._jsc.hadoopConfiguration() 设置 GCS 认证
VOLUME_BASE = "/Volumes/workspace/default/olist_files"

# CSV 文件名 → Bronze Delta 表名的映射字典
# 命名规范：所有 Bronze 表统一加 bronze_ 前缀，便于后续 Silver 层引用
# 设计决策：用字典而非逐个手写调用，方便增删表和维护
# 注意：这里只做格式转换，列名和列类型完全保留原始状态
TABLE_MAPPING = {
    "olist_orders_dataset.csv":              "bronze_orders",
    "olist_customers_dataset.csv":           "bronze_customers",
    "olist_order_items_dataset.csv":         "bronze_order_items",
    "olist_products_dataset.csv":            "bronze_products",
    "olist_sellers_dataset.csv":             "bronze_sellers",
    "olist_order_payments_dataset.csv":      "bronze_payments",
    "olist_order_reviews_dataset.csv":       "bronze_reviews",
    "olist_geolocation_dataset.csv":         "bronze_geolocation",
    "product_category_name_translation.csv": "bronze_translation",
}

print(f"📂 数据源: {VOLUME_BASE}")
print(f"📋 待导入表数: {len(TABLE_MAPPING)}")

In [0]:
# ============================================================
# 核心函数：单张 CSV → Delta Bronze 表
# ============================================================

def ingest_csv_to_bronze(file_name, table_name):
    """
    从数据源读取单个 CSV 文件，写入 Databricks 的 Delta Lake Bronze 表。

    参数:
        file_name:  CSV 文件名（如 "olist_orders_dataset.csv"）
        table_name: 目标 Delta 表名（如 "bronze_orders"）

    设计决策:
        - header="true": CSV 第一行是列名，不是数据
        - inferSchema="true": 让 Spark 自动推断列类型
          （Bronze 层保留原始数据面貌，不手动 CAST）
        - mode("overwrite"): 全量覆盖 — Olist 是静态快照，无增量需求
        - 写入 Delta 格式而非 Parquet，因为 Delta 额外提供：
          ① ACID 事务保证（不会写一半失败留下脏数据）
          ② 时间旅行（可回溯历史版本）
          ③ Schema 演化（后续加列不会报错）
    """
    # 拼接完整的数据源路径
    full_path = f"{VOLUME_BASE}/{file_name}"
    print(f"📥 正在读取: {full_path}")

    # Spark CSV Reader：读取 CSV，自动推断 Schema，不做任何清洗
    df = spark.read.format("csv") \
        .option("header", "true") \
        .option("inferSchema", "true") \
        .load(full_path)

    # 写入 Delta Lake 表
    # mode("overwrite") — Olist 是静态快照，每次全量覆盖即可
    df.write.format("delta").mode("overwrite").saveAsTable(table_name)

    # 打印行数，确认导入成功
    row_count = df.count()
    print(f"✅ {table_name} 写入完成 — {row_count:,} 行")

In [0]:
# ============================================================
# 批量执行：遍历 9 张表，依次读入并写入 Bronze 层
# ============================================================

# 使用 for 循环批量调用 ingest 函数
# - 比逐个手写调用更易维护（新增/删除表只需改 TABLE_MAPPING 字典）
# - 每张表的导入相互独立，一张失败不影响其他表
#   （如需更强的容错，可在循环内加 try/except）
for file_name, table_name in TABLE_MAPPING.items():
    ingest_csv_to_bronze(file_name, table_name)

print("\n🎉 全部 9 张 Bronze 表写入完成！")

In [0]:
# ============================================================
# 验证：列出所有新创建的 Bronze 表，确认数量和名称
# ============================================================

# 查询 hive_metastore 的 default schema 下所有 bronze_ 开头的表
# 预期输出：9 张表，从 bronze_orders 到 bronze_translation
print("📋 Bronze 层已注册的表：")
spark.sql("SHOW TABLES IN default LIKE 'bronze_*'").show(20, False)

---

### 下一步

Bronze 表创建完成后，进入 02_silver 做数据清洗和 JOIN。